# `utils_MemberB.ipynb` — Edit-distance features

**Owner: Member B.**  This notebook documents section *3. MEMBER B — Edit-distance features* of `utils.py`.

The core function is `levenshtein_distance`, implemented from scratch with a memory-efficient two-row dynamic-programming table; everything else is a thin wrapper that exposes the algorithm at different granularities (characters vs. tokens, raw vs. normalised).

In [ ]:
import utils

Q1 = 'How can I learn Python quickly?'
Q2 = 'How can I learn Python fast?'
Q3 = 'What is the boiling point of water?'

## `levenshtein_distance(s, t)`

Standard edit distance: minimum number of single-character (or single-token) **insertions, deletions and substitutions** to turn `s` into `t`.

**Recurrence.**  Let `d[i][j]` be the edit distance between `s[:i]` and `t[:j]`. Then
$$d[i][j] = \min\bigl(d[i-1][j] + 1,\; d[i][j-1] + 1,\; d[i-1][j-1] + \mathbb{1}[s_i \ne t_j]\bigr).$$

Our implementation uses **two rolling rows** instead of a full $(m+1) \times (n+1)$ matrix, dropping memory from $O(mn)$ to $O(\min(m, n))$.  The function works on any indexable sequence, so we can reuse it for both strings and token lists.

In [ ]:
# Sanity checks against a hand-crafted reference implementation.
def ref(s, t):
    if not s: return len(t)
    if not t: return len(s)
    dp = [[0]*(len(t)+1) for _ in range(len(s)+1)]
    for i in range(len(s)+1): dp[i][0] = i
    for j in range(len(t)+1): dp[0][j] = j
    for i in range(1, len(s)+1):
        for j in range(1, len(t)+1):
            cost = 0 if s[i-1] == t[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return dp[len(s)][len(t)]

for a, b in [('kitten','sitting'), ('','abc'), ('abc',''), ('flaw','lawn'), ('abcdef','azced')]:
    assert utils.levenshtein_distance(a, b) == ref(a, b)
print('Levenshtein matches the reference implementation on all test cases.')

## Character-level features

* `char_edit_distance(q1, q2)` — raw distance.
* `normalised_char_edit(q1, q2)` — distance divided by the length of the longer string, in $[0, 1]$.  This is the *characters-changed* fraction and is more useful for the classifier than the raw count, which scales with sentence length.

In [ ]:
print('char edit (paraphrase) :', utils.char_edit_distance(Q1, Q2))
print('char edit (unrelated)  :', utils.char_edit_distance(Q1, Q3))
print('normalised paraphrase  :', round(utils.normalised_char_edit(Q1, Q2), 3))
print('normalised unrelated   :', round(utils.normalised_char_edit(Q1, Q3), 3))

## Token-level features

Same algorithm but applied to *token* lists.  This catches a different kind of similarity: replacing one word by another counts as a single edit, regardless of how different the two words are spelled.

* `token_edit_distance` — raw
* `normalised_token_edit` — divided by max(len_in_tokens), in $[0, 1]$

In [ ]:
print('token edit (paraphrase):', utils.token_edit_distance(Q1, Q2))
print('token edit (unrelated) :', utils.token_edit_distance(Q1, Q3))

## `longest_common_prefix(q1, q2)`

Number of leading tokens that are identical.  Many Quora duplicates start with the same template (`"How can I ..."`, `"What is the best way to ..."`), so a long shared prefix is a cheap and informative feature.

In [ ]:
print(utils.longest_common_prefix(Q1, Q2))   # 'How can I learn Python' -> 5
print(utils.longest_common_prefix(Q1, Q3))   # 0

## Trade-offs

* **Strength.**  Edit distance is *order-aware*, unlike Jaccard.  `"the cat sat on the mat"` vs `"the mat sat on the cat"` have Jaccard 1.0 but a non-trivial token edit distance.
* **Weakness.**  It treats every substitution as equally costly: substituting `"car"` by `"automobile"` costs the same as substituting `"car"` by `"the"`.  Semantic substitutions are handled instead by Member C's TF-IDF cosine and (in future iterations) by word-embedding similarity.